## Model Training

In [30]:
import pandas as pd
df=pd.read_csv('data/default_of_credit_card_clients.csv')

In [2]:
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [31]:
# Lets drop ID column as it is not required
df=df.drop(columns='ID')

In [33]:
# Rename the original column without changing its position
df = df.rename(columns={"PAY_0": "PAY_1"})

In [35]:
#dropping duplicates
df = df.drop_duplicates(keep="first", ignore_index=True)

In [36]:
# Dropping correlated features
df=df.drop(labels=['BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6'],axis=1)

In [37]:
# Independent and dependetnt features segregation
X=df.drop(labels=['default payment next month'],axis=1)
y=df[['default payment next month']]

In [38]:
repayment_columns=['PAY_1','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']

In [39]:
# Segregate numerical and categorical columns
categorical_columns=['SEX','EDUCATION','MARRIAGE']+repayment_columns
numerical_columns=[
    col for col in X.columns if col not in categorical_columns
]


In [17]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [40]:
##Numerical_pipeline
num_pipeline=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',StandardScaler())
    ]
)

##Categorical Pipeline
cat_pipeline=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(drop="if_binary",handle_unknown="ignore")), # no scaling required for OneHot encoding
         
    ]
)
preprocessor=ColumnTransformer([
('num pipeline',num_pipeline,numerical_columns),
('cat pipeline',cat_pipeline,categorical_columns)
])

In [50]:
## Train test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

In [51]:
X_train.shape,X_test.shape

((20975, 17), (8990, 17))

In [ ]:
preprocessor.fit_transform(X_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 348257 stored elements and shape (20975, 82)>

In [53]:
pd.DataFrame.sparse.from_spmatrix(preprocessor.fit_transform(X_train),columns=preprocessor.get_feature_names_out())

,num pipeline__LIMIT_BAL,num pipeline__AGE,num pipeline__PAY_AMT1,num pipeline__PAY_AMT2,num pipeline__PAY_AMT3,num pipeline__PAY_AMT4,num pipeline__PAY_AMT5,num pipeline__PAY_AMT6,cat pipeline__SEX_2,cat pipeline__EDUCATION_0,...,cat pipeline__PAY_6_-2,cat pipeline__PAY_6_-1,cat pipeline__PAY_6_0,cat pipeline__PAY_6_2,cat pipeline__PAY_6_3,cat pipeline__PAY_6_4,cat pipeline__PAY_6_5,cat pipeline__PAY_6_6,cat pipeline__PAY_6_7,cat pipeline__PAY_6_8
0,-0.66844,0.383239,-0.28726,-0.037545,-0.128805,-0.162988,-0.15622,-0.297835,NaN,NaN,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN
1,-0.900566,-1.13611,-0.247706,-0.150815,-0.159107,-0.312081,-0.312009,-0.297835,1.0,NaN,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-1.132692,-1.13611,-0.264247,-0.144208,-0.220057,-0.272799,-0.279553,-0.240292,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2.581323,0.600289,-0.195206,-0.265361,-0.301959,-0.260223,-0.312009,-0.211521,NaN,NaN,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN
4,1.80757,0.166189,4.742395,0.645902,5.065794,1.632616,0.986228,0.393482,1.0,NaN,...,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20970,0.105313,-0.159386,-0.365716,-0.265361,-0.301959,-0.312081,-0.312009,-0.297835,NaN,NaN,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20971,-0.591065,-0.810535,-0.234957,-0.128671,-0.186523,-0.312081,-0.091309,-0.297835,NaN,NaN,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN
20972,-0.900566,-1.027585,-0.365716,-0.265361,-0.301959,-0.312081,-0.312009,-0.297835,NaN,NaN,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20973,0.879066,-1.13611,0.045195,0.003325,0.038288,0.190104,0.110502,0.348599,1.0,NaN,...,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)

c:\Data Science course\ML Projects\CreditCardDefaulters\CreditCardDefaulters\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [6, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## Model Training

In [57]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score

In [58]:
LogReg=LogisticRegression()
LogReg.fit(X_train,y_train)

c:\Data Science course\ML Projects\CreditCardDefaulters\CreditCardDefaulters\Lib\site-packages\sklearn\utils\validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [59]:
LogReg.coef_

array([[-0.17762069,  0.02398304, -0.12974858, -0.14529512, -0.03005203,
        -0.05926078, -0.0452339 , -0.04816869, -0.13179862, -0.34309797,
         0.34586764,  0.3994259 ,  0.32018634, -0.44760588, -0.95503741,
         0.35376812, -0.96999958,  0.34806586,  0.16923926,  0.1262012 ,
        -1.24753846, -0.65062979, -1.25184976, -0.33336507,  0.97650999,
         0.95880227,  0.8567832 ,  0.20851996, -0.20464839,  0.24534724,
         0.11557554,  0.12639542, -0.23115741, -0.02108133, -0.52456391,
         0.07062627,  0.09560449, -0.76362521,  0.31583941,  0.47326824,
         0.25290979, -0.12070901, -0.34294653, -0.33328183, -0.29079327,
         0.14923045,  0.07945414,  0.22776147, -0.1053086 ,  0.09235817,
         0.2416524 , -0.10587584,  0.06125619, -0.19958672, -0.28176261,
        -0.22533974,  0.24043579,  0.10639809, -0.0067535 ,  0.54656908,
        -0.41359916, -0.19721796,  0.10436347, -0.08248234, -0.11509133,
        -0.02939167,  0.11138436, -0.09807404, -0.4

In [60]:
LogReg.intercept_

array([-0.44629905])

In [88]:
def evaluate_model(true,predicted):
    acc_score=accuracy_score(true,predicted)
    f1=f1_score(true,predicted)
    report=classification_report(true,predicted,labels=[0,1],target_names=['Non-Defaulters','Defaulters'],zero_division=0)
    matrix=confusion_matrix(true,predicted,labels=[0,1])

    return acc_score,f1,report,matrix


In [94]:
# Training multiple models
models={
    "Logistic Regression":LogisticRegression(),
    "SVC":SVC(),
    "Decision Tree ":DecisionTreeClassifier(),
    "Random Forest Classifier":RandomForestClassifier()
}

model_list=list()
results=list()

for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,y_train)

    #Make prediction
    y_pred=model.predict(X_test)
    acc_score,f1,report,matrix=evaluate_model(y_test,y_pred)
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    results.append({"Model Name": list(models.keys())[i],
                   "Accuracy": acc_score,
                   "F1 Score": f1})
    
    print('Model Performance')
    print(f"Accuracy Score: {acc_score:.4f}")
    print(f"f1 Score: {f1:.4f}")
    print('Classification Report') 
    print(report)
    print('Confusion Matrix')
    print(matrix)
    print("*"*60)
    print('\n')

results_df = pd.DataFrame(results)
display(results_df)

c:\Data Science course\ML Projects\CreditCardDefaulters\CreditCardDefaulters\Lib\site-packages\sklearn\utils\validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Logistic Regression
Model Performance
Accuracy Score: 0.8179
f1 Score: 0.4552
Classification Report
                precision    recall  f1-score   support

Non-Defaulters       0.83      0.96      0.89      6978
    Defaulters       0.69      0.34      0.46      2012

      accuracy                           0.82      8990
     macro avg       0.76      0.65      0.67      8990
  weighted avg       0.80      0.82      0.79      8990

Confusion Matrix
[[6669  309]
 [1328  684]]
************************************************************




c:\Data Science course\ML Projects\CreditCardDefaulters\CreditCardDefaulters\Lib\site-packages\sklearn\utils\validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


SVC
Model Performance
Accuracy Score: 0.8187
f1 Score: 0.4379
Classification Report
                precision    recall  f1-score   support

Non-Defaulters       0.83      0.96      0.89      6978
    Defaulters       0.72      0.32      0.44      2012

      accuracy                           0.82      8990
     macro avg       0.77      0.64      0.66      8990
  weighted avg       0.80      0.82      0.79      8990

Confusion Matrix
[[6725  253]
 [1377  635]]
************************************************************


Decision Tree 
Model Performance
Accuracy Score: 0.7263
f1 Score: 0.4011
Classification Report
                precision    recall  f1-score   support

Non-Defaulters       0.83      0.82      0.82      6978
    Defaulters       0.39      0.41      0.40      2012

      accuracy                           0.73      8990
     macro avg       0.61      0.61      0.61      8990
  weighted avg       0.73      0.73      0.73      8990

Confusion Matrix
[[5705 1273]
 [1188

c:\Data Science course\ML Projects\CreditCardDefaulters\CreditCardDefaulters\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Random Forest Classifier
Model Performance
Accuracy Score: 0.8147
f1 Score: 0.4629
Classification Report
                precision    recall  f1-score   support

Non-Defaulters       0.84      0.95      0.89      6978
    Defaulters       0.66      0.36      0.46      2012

      accuracy                           0.81      8990
     macro avg       0.75      0.65      0.68      8990
  weighted avg       0.80      0.81      0.79      8990

Confusion Matrix
[[6606  372]
 [1294  718]]
************************************************************




,Model Name,Accuracy,F1 Score
0,Logistic Regression,0.817909,0.455241
1,SVC,0.818687,0.437931
2,Decision Tree,0.726251,0.401071
3,Random Forest Classifier,0.814683,0.462927
